In [1]:
# Configuration
PROJECT_ID = "project-b38b370e-25fb-420a-a54"
LOCATION = "us-central1"

# Endpoint IDs
V1_ENDPOINT_ID = "2976708379633778688"
V2_ENDPOINT_ID = "6201707925296119808"

print("Config set!")

Config set!


In [2]:
# Install
!pip install google-cloud-aiplatform -q

# Import
import pandas as pd
import json
from google.cloud import aiplatform

# Initialize Vertex AI
aiplatform.init(project=PROJECT_ID, location=LOCATION)

print("Libraries imported and Vertex AI initialized!")

Libraries imported and Vertex AI initialized!


/opt/micromamba/lib/python3.12/site-packages/google/cloud/aiplatform/models.py:52: FutureWarning: Support for google-cloud-storage < 3.0.0 will be removed in a future version of google-cloud-aiplatform. Please upgrade to google-cloud-storage >= 3.0.0.
  from google.cloud.aiplatform.utils import gcs_utils


In [3]:
import os
# Find where iris_test.csv is
result = os.popen('find /home/jupyter -name "iris_test.csv"').read()
print(result)

/home/jupyter/iris_test.csv



In [4]:
# Load proper test data we saved earlier
test_df = pd.read_csv('/home/jupyter/iris_test.csv')
test_df = test_df.reset_index(drop=True)

print(f"Test samples: {len(test_df)}")
print(test_df.head())

Test samples: 21
   sepal_length  sepal_width  petal_length  petal_width     species
0           4.9          3.1           1.5          0.1      setosa
1           6.1          3.0           4.6          1.4  versicolor
2           5.2          4.1           1.5          0.1      setosa
3           7.7          2.8           6.7          2.0   virginica
4           4.4          3.2           1.3          0.2      setosa


In [5]:
from google.cloud import aiplatform

def predict_species(endpoint_id, input_text):
    try:
        endpoint = aiplatform.Endpoint(endpoint_id)
        instance = {"content": input_text}
        response = endpoint.predict(instances=[instance])
        try:
            return response.predictions[0]['content'].strip()
        except:
            return str(response.predictions[0]).strip()
    except Exception as e:
        print(f"Error: {e}")
        return "error"

print("Prediction function ready!")

Prediction function ready!


In [8]:
import vertexai
from vertexai.generative_models import GenerativeModel

vertexai.init(project=PROJECT_ID, location=LOCATION)

# Use tuned model endpoint directly
def predict_v1(row):
    model = GenerativeModel(
        model_name="gemini-2.5-flash-lite",
        system_instruction="You are an iris flower classifier. Classify the iris species based on measurements. Reply with only the species name: setosa, versicolor, or virginica."
    )
    input_text = f"sepal_length: {row['sepal_length']}, sepal_width: {row['sepal_width']}, petal_length: {row['petal_length']}, petal_width: {row['petal_width']}"
    response = model.generate_content(input_text)
    return response.text.strip().lower()

# Test with first sample
test_row = test_df.iloc[0]
result = predict_v1(test_row)
print(f"Predicted: {result}")
print(f"Actual: {test_row['species']}")

Predicted: setosa
Actual: setosa


In [9]:
import vertexai
from vertexai.tuning import sft

vertexai.init(project=PROJECT_ID, location=LOCATION)

# List all tuning jobs
tuning_jobs = sft.SupervisedTuningJob.list()
for job in tuning_jobs:
    print(f"Name: {job.name}")
    print(f"State: {job.state}")
    print(f"Tuned model: {job.tuned_model_name}")
    print("---")

Name: 708989432874663936
State: 4
Tuned model: projects/927061930480/locations/us-central1/models/2444311655325106176@1
---
Name: 5870114605841252352
State: 4
Tuned model: projects/927061930480/locations/us-central1/models/1753009112523735040@1
---
Name: 5450153940588953600
State: 5
Tuned model: 
---


In [19]:
import requests
import google.auth
import google.auth.transport.requests

def predict_with_endpoint(endpoint_id, input_text):
    credentials, project = google.auth.default()
    auth_req = google.auth.transport.requests.Request()
    credentials.refresh(auth_req)
    token = credentials.token
    
    url = f"https://us-central1-aiplatform.googleapis.com/v1/projects/927061930480/locations/us-central1/endpoints/{endpoint_id}:generateContent"
    
    headers = {
        "Authorization": f"Bearer {token}",
        "Content-Type": "application/json"
    }
    
    payload = {
        "contents": [
            {"role": "user", "parts": [{"text": input_text}]}
        ],
        "generationConfig": {
            "temperature": 0.0
        }
    }
    
    response = requests.post(url, headers=headers, json=payload)
    result = response.json()
    
    try:
        text = result['candidates'][0]['content']['parts'][0]['text'].strip().lower()
        # Extract species name
        for species in ['setosa', 'versicolor', 'virginica']:
            if species in text:
                return species
        return text
    except:
        return "error"

# Test V2 endpoint
test_row = test_df.iloc[0]
input_text = f"A flower specimen has a sepal length of {test_row['sepal_length']} cm, sepal width of {test_row['sepal_width']} cm, petal length of {test_row['petal_length']} cm, and petal width of {test_row['petal_width']} cm. Identify the iris species."
result = predict_with_endpoint("6201707925296119808", input_text)
print(f"Predicted: {result}")
print(f"Actual: {test_row['species']}")

Predicted: setosa
Actual: setosa


In [20]:
# Check all endpoints
response = requests.get(
    "https://us-central1-aiplatform.googleapis.com/v1/projects/927061930480/locations/us-central1/endpoints",
    headers={"Authorization": f"Bearer {token}", "Content-Type": "application/json"}
)

import json
endpoints = response.json()['endpoints']
for ep in endpoints:
    print(f"Name: {ep['name']}")
    print(f"Display: {ep['displayName']}")
    print("---")

Name: projects/927061930480/locations/us-central1/endpoints/6201707925296119808
Display: 	iris-v2-natural
---
Name: projects/927061930480/locations/us-central1/endpoints/4909174832240787456
Display: 	iris-v2-natural
---
Name: projects/927061930480/locations/us-central1/endpoints/2238540253210083328
Display: 	iris-v2-natural
---
Name: projects/927061930480/locations/us-central1/endpoints/2976708379633778688
Display: iris-v1-raw
---
Name: projects/927061930480/locations/us-central1/endpoints/5044282821061902336
Display: iris-v1-raw
---
Name: projects/927061930480/locations/us-central1/endpoints/5999045942064447488
Display: iris-v1-raw
---


In [26]:
# Endpoint IDs
V1_ENDPOINT = "2976708379633778688"
V2_ENDPOINT = "6201707925296119808"

# Run V1 predictions
print("Running V1 predictions...")
v1_predictions = []
for _, row in test_df.iterrows():
    input_text = f"sepal_length: {row['sepal_length']}, sepal_width: {row['sepal_width']}, petal_length: {row['petal_length']}, petal_width: {row['petal_width']}. Reply with only one word: setosa, versicolor, or virginica."
    pred = predict_with_endpoint(V1_ENDPOINT, input_text)
    v1_predictions.append(pred)
    print(f"Actual: {row['species']} → Predicted: {pred}")

print(f"\nV1 Done! Total: {len(v1_predictions)}")

Running V1 predictions...
Actual: setosa → Predicted: setosa
Actual: versicolor → Predicted: versicolor
Actual: setosa → Predicted: setosa
Actual: virginica → Predicted: virginica
Actual: setosa → Predicted: setosa
Actual: virginica → Predicted: virginica
Actual: virginica → Predicted: virginica
Actual: setosa → Predicted: setosa
Actual: virginica → Predicted: virginica
Actual: setosa → Predicted: setosa
Actual: versicolor → Predicted: virginica
Actual: virginica → Predicted: virginica
Actual: versicolor → Predicted: error
Actual: virginica → Predicted: virginica
Actual: setosa → Predicted: setosa
Actual: versicolor → Predicted: error
Actual: setosa → Predicted: error
Actual: versicolor → Predicted: versicolor
Actual: virginica → Predicted: error
Actual: versicolor → Predicted: versicolor
Actual: setosa → Predicted: error

V1 Done! Total: 21


In [22]:
print(f"Total V1 predictions collected: {len(v1_predictions)}")
print(f"Total test samples: {len(test_df)}")

Total V1 predictions collected: 21
Total test samples: 21


In [25]:
# Run V2 predictions
print("Running V2 predictions...")
v2_predictions = []
for _, row in test_df.iterrows():
    input_text = f"A flower specimen has a sepal length of {row['sepal_length']} cm, sepal width of {row['sepal_width']} cm, petal length of {row['petal_length']} cm, and petal width of {row['petal_width']} cm. Identify the iris species. Reply with only one word: setosa, versicolor, or virginica."
    pred = predict_with_endpoint(V2_ENDPOINT, input_text)
    v2_predictions.append(pred)
    print(f"Actual: {row['species']} → Predicted: {pred}")

print(f"\nV2 Done! Total: {len(v2_predictions)}")

Running V2 predictions...
Actual: setosa → Predicted: setosa
Actual: versicolor → Predicted: setosa
Actual: setosa → Predicted: setosa
Actual: virginica → Predicted: virginica
Actual: setosa → Predicted: setosa
Actual: virginica → Predicted: virginica
Actual: virginica → Predicted: setosa
Actual: setosa → Predicted: setosa
Actual: virginica → Predicted: virginica
Actual: setosa → Predicted: setosa
Actual: versicolor → Predicted: setosa
Actual: virginica → Predicted: virginica
Actual: versicolor → Predicted: error
Actual: virginica → Predicted: setosa
Actual: setosa → Predicted: error
Actual: versicolor → Predicted: setosa
Actual: setosa → Predicted: error
Actual: versicolor → Predicted: setosa
Actual: virginica → Predicted: setosa
Actual: versicolor → Predicted: setosa
Actual: setosa → Predicted: setosa

V2 Done! Total: 21


In [27]:
from sklearn.metrics import accuracy_score, classification_report

# Valid species
valid_species = ['setosa', 'versicolor', 'virginica']
actual = list(test_df['species'])

# Format compliance rate
v1_compliance = [p for p in v1_predictions if p in valid_species]
v2_compliance = [p for p in v2_predictions if p in valid_species]

v1_compliance_rate = len(v1_compliance) / len(v1_predictions) * 100
v2_compliance_rate = len(v2_compliance) / len(v2_predictions) * 100

# Clean predictions (replace error with unknown)
v1_clean = [p if p in valid_species else 'unknown' for p in v1_predictions]
v2_clean = [p if p in valid_species else 'unknown' for p in v2_predictions]

# Accuracy
v1_accuracy = accuracy_score(actual, v1_clean) * 100
v2_accuracy = accuracy_score(actual, v2_clean) * 100

print("=" * 50)
print("       EVALUATION RESULTS")
print("=" * 50)

print(f"\nV1 (Raw Format):")
print(f"  Accuracy:          {v1_accuracy:.1f}%")
print(f"  Format Compliance: {v1_compliance_rate:.1f}%")

print(f"\nV2 (Natural Language):")
print(f"  Accuracy:          {v2_accuracy:.1f}%")
print(f"  Format Compliance: {v2_compliance_rate:.1f}%")

print("\n--- V1 Classification Report ---")
print(classification_report(actual, v1_clean, zero_division=0))

print("\n--- V2 Classification Report ---")
print(classification_report(actual, v2_clean, zero_division=0))

print("=" * 50)
if v1_accuracy > v2_accuracy:
    print("✅ V1 (Raw Format) performs better!")
elif v2_accuracy > v1_accuracy:
    print("✅ V2 (Natural Language) performs better!")
else:
    print("Both models perform equally!")

       EVALUATION RESULTS

V1 (Raw Format):
  Accuracy:          71.4%
  Format Compliance: 76.2%

V2 (Natural Language):
  Accuracy:          47.6%
  Format Compliance: 85.7%

--- V1 Classification Report ---
              precision    recall  f1-score   support

      setosa       1.00      0.75      0.86         8
     unknown       0.00      0.00      0.00         0
  versicolor       1.00      0.50      0.67         6
   virginica       0.86      0.86      0.86         7

    accuracy                           0.71        21
   macro avg       0.71      0.53      0.60        21
weighted avg       0.95      0.71      0.80        21


--- V2 Classification Report ---
              precision    recall  f1-score   support

      setosa       0.43      0.75      0.55         8
     unknown       0.00      0.00      0.00         0
  versicolor       0.00      0.00      0.00         6
   virginica       1.00      0.57      0.73         7

    accuracy                           0.48      